<a href="https://colab.research.google.com/github/RushabhMowade/Nutrient-Deficiency-and-Growth-Stage-Prediction/blob/main/coffee_aug_seeedling_gs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install bing_image_downloader

In [3]:
from bing_image_downloader import downloader
import os
import shutil

# --- SETTINGS ---
CONFIRMED_PATH = '/content/drive/MyDrive/nutrient-deficiency/coffee_growth_stage/seedling'
SEARCH_QUERY = "coffee plant chapola seedling nursery"
LIMIT_COUNT = 300 # Getting a few extra to account for "junk" images

# Create folder
if not os.path.exists(CONFIRMED_PATH):
    os.makedirs(CONFIRMED_PATH)

# Download to a temporary spot
downloader.download(
    SEARCH_QUERY,
    limit=LIMIT_COUNT,
    output_dir='/content/temp_coffee',
    adult_filter_off=True,
    force_replace=False,
    timeout=60,
    verbose=False
)

# Move to Drive
temp_folder = f'/content/temp_coffee/{SEARCH_QUERY}'
downloaded_files = os.listdir(temp_folder)

for i, filename in enumerate(downloaded_files):
    # Renaming them so they are easy to identify in your Master List
    new_name = f"coffee_seedling_new_{i}.jpg"
    shutil.move(os.path.join(temp_folder, filename), os.path.join(CONFIRMED_PATH, new_name))

print(f"✅ Finished! {len(downloaded_files)} Coffee Seedlings moved to {CONFIRMED_PATH}")

[%] Downloading Images to /content/temp_coffee/coffee plant chapola seedling nursery
[!] Issue getting: https://c.stocksy.com/a/qbyI00/z9/4522890.jpg
[!] Error:: HTTP Error 403: Forbidden
[!] Issue getting: https://media.gettyimages.com/id/1173237923/photo/a-man-checks-coffee-plant-seedlings-at-los-papales-farm-in-jinotega-nicaragua-on-august-26.jpg?s=612x612&amp;w=gi&amp;k=20&amp;c=6DQshJCBf80yKwtzRi5DWAg0DdTe242msHm8-VPJfQQ=
[!] Error:: HTTP Error 400: Bad Request
[!] Issue getting: https://media.gettyimages.com/id/1173237925/photo/a-man-shows-a-coffee-plant-seedling-at-los-papales-farm-in-jinotega-nicaragua-on-august-26.jpg?s=612x612&amp;w=gi&amp;k=20&amp;c=xGuxIjW0X9bBq8VoHP7IhOwZHQZIgN80x8QIFbNXIGE=
[!] Error:: HTTP Error 400: Bad Request
[!] Issue getting: https://media.gettyimages.com/id/1858869870/photo/coffee-seedlings-in-the-greenhouse-at-the-western-highlands-agro-forestry-scientific.jpg?s=612x612&amp;w=gi&amp;k=20&amp;c=46JrsoHwJKucV-l3z3P0kBEuZLkYxSCqwP1Abytgp_w=
[!] Error

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img

# --- PATH SETUP ---
COFFEE_BASE = '/content/drive/MyDrive/nutrient-deficiency/coffee_growth_stage'
# Mapping: 1=seedling, 2=vegetative, 3=flowering
target_count = 1000

# --- THE "NICE" AUGMENTATION STRATEGY ---
datagen = ImageDataGenerator(
    rotation_range=40,         # Randomly rotate (helps with different plant orientations)
    width_shift_range=0.2,     # Horizontal shift
    height_shift_range=0.2,    # Vertical shift
    shear_range=0.2,           # Slants the image (simulates perspective)
    zoom_range=0.3,            # Randomly zooms in/out (VERY important for growth stages)
    horizontal_flip=True,      # Flips the image (doubles your data instantly)
    vertical_flip=True,        # Plants can be viewed from any side
    brightness_range=[0.6, 1.4], # Simulates shadows vs. bright sunlight
    fill_mode='reflect'        # Replaces empty pixels with a reflection (looks more natural)
)

for stage in ['1', '2', '3']:
    folder_path = os.path.join(COFFEE_BASE, stage)
    if not os.path.exists(folder_path):
        continue

    # Get only original images (ignore previous augmented ones if any)
    originals = [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg')) and 'aug_' not in f]
    current_total = len(os.listdir(folder_path))
    num_to_generate = target_count - current_total

    if num_to_generate <= 0:
        print(f"✅ Stage {stage} already satisfied ({current_total} images).")
        continue

    print(f"🚀 Enhancing Stage {stage}: Adding {num_to_generate} high-quality variations...")

    gen_count = 0
    while gen_count < num_to_generate:
        for filename in originals:
            if gen_count >= num_to_generate:
                break

            try:
                img = load_img(os.path.join(folder_path, filename))
                x = img_to_array(img)
                x = x.reshape((1,) + x.shape)

                # Save the new version
                for batch in datagen.flow(x, batch_size=1,
                                          save_to_dir=folder_path,
                                          save_prefix=f'aug_stage{stage}',
                                          save_format='jpeg'):
                    gen_count += 1
                    break
            except:
                continue

    print(f"✨ Stage {stage} complete: Total 1,000 images.")

🚀 Enhancing Stage 1: Adding 774 high-quality variations...
✨ Stage 1 complete: Total 1,000 images.
🚀 Enhancing Stage 2: Adding 204 high-quality variations...
✨ Stage 2 complete: Total 1,000 images.
🚀 Enhancing Stage 3: Adding 559 high-quality variations...
